# ANN Data Preparation

Load the cleaned dataset, create stratified splits, and preprocess features. The next step is to wrap these splits in a PyTorch `Dataset` and `DataLoader`.

## Load and inspect data

Load the cleaned dataset and review a sample and its schema.

In [19]:
import re



def drive_to_direct_url(link: str) -> str:
    """
    Convert Google Drive share link to direct download URL.
    Works for multiple link formats.
    """
    patterns = [
        r"/file/d/([a-zA-Z0-9_-]+)",
        r"id=([a-zA-Z0-9_-]+)",
        r"/open\?id=([a-zA-Z0-9_-]+)",
    ]

    file_id = None

    for pattern in patterns:
        match = re.search(pattern, link)
        if match:
            file_id = match.group(1)
            break

    if not file_id:
        raise ValueError("Invalid Google Drive link")

    return f"https://drive.google.com/uc?id={file_id}"

In [20]:
clean_dataset_drive_url = (
    "https://drive.google.com/file/d/1AxchD_W4q0d9lXBOi2gyqunl_TmH3ktm/view?usp=sharing"
)
dataset = drive_to_direct_url(clean_dataset_drive_url)

In [21]:
import pandas as pd

df = pd.read_csv(dataset)

In [22]:
df.head()

,age,gender,pulse_rate,systolic_bp,diastolic_bp,glucose,bmi,family_diabetes,hypertensive,family_hypertension,cardiovascular_disease,stroke,diabetic
0,42,Female,66,110,73,5.88,25.75,0,0,0,0,0,No
1,35,Female,60,125,68,5.71,19.58,0,0,0,0,0,No
2,62,Female,57,127,74,6.85,20.24,0,0,0,0,0,No
3,73,Male,55,193,112,6.28,21.72,0,0,0,0,0,No
4,68,Female,71,150,81,5.71,17.79,0,0,0,0,0,No


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5261 entries, 0 to 5260
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   age                     5261 non-null   int64  
 1   gender                  5261 non-null   object 
 2   pulse_rate              5261 non-null   int64  
 3   systolic_bp             5261 non-null   int64  
 4   diastolic_bp            5261 non-null   int64  
 5   glucose                 5261 non-null   float64
 6   bmi                     5261 non-null   float64
 7   family_diabetes         5261 non-null   int64  
 8   hypertensive            5261 non-null   int64  
 9   family_hypertension     5261 non-null   int64  
 10  cardiovascular_disease  5261 non-null   int64  
 11  stroke                  5261 non-null   int64  
 12  diabetic                5261 non-null   object 
dtypes: float64(2), int64(9), object(2)
memory usage: 534.4+ KB


## Split and encode

Separate the target, encode its labels, and create stratified train, validation, and test splits.

In [24]:
X = df.drop(columns="diabetic")
y = df["diabetic"]

In [25]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

In [26]:
from sklearn.model_selection import train_test_split


X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

## Preprocess features

Fit scaling and categorical encoding on the training split, then apply them to validation and test data.

In [27]:
numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)


In [29]:
X_train = preprocessor.fit_transform(X_train)
X_val = preprocessor.transform(X_val)
X_test = preprocessor.transform(X_test)

## Next step

Implement the PyTorch `Dataset` and `DataLoader` using the prepared splits.